# 10.5 Beyond Relational - Key-Value and Document Stores

**Prerequisites:** 10.1 Introduction to SQL, 10.3 SQLite in Python  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- When a relational table stops being the right shape
- The four families of non-relational store, and what each is actually for
- **Key-value stores in the standard library** - `dbm.sqlite3` and `shelve`
- 🔴 The `shelve` **writeback trap**, which loses writes silently
- Redis: TTLs, atomic counters and hashes
- **Document stores**, and schema-on-read vs schema-on-write
- Building a real document store on **SQLite JSON1** - and indexing a JSON path
- **PostgreSQL JSONB**: containment queries and GIN indexes
- MongoDB from Python, and TinyDB when you have no server
- 🔴 What you give up: the constraints that were doing work for you
- Eventual consistency and CAP, in plain English
- DuckDB - columnar and analytical, still in-process

---

## The problem with one shape

Folders 10.1 to 10.4 taught one model: **rows and columns, defined up front**. It is the right default and it is not going anywhere. But it assumes three things that are not always true:

1. **You know the columns in advance.** A `webhook_event` table receiving payloads from forty different providers does not.
2. **Every row has the same shape.** Product catalogues are the classic counter-example: a book has an ISBN, a T-shirt has a size, and neither has the other's column.
3. **A join is affordable.** At three hops and a few million rows it usually is. At eight hops it usually is not - which is **10.6**.

### An analogy

A relational database is a **filing cabinet with pre-cut dividers**. Everything fits beautifully, provided it fits. A document store is a **shelf of labelled folders**: each folder holds whatever that thing needs, and it is your job to remember what should be inside. You gain flexibility and you lose the guarantee.

> Non-relational does **not** mean better. It means *different trade-offs*. Most applications should start relational and reach for these when they have a specific reason, not before.

## The four families

| Family | Shape | Good at | Bad at | Examples |
|---|---|---|---|---|
| **Key-value** | `key -> blob` | Fast lookup by exact key; caching; sessions | Any query that is not by key | Redis, Memcached, `dbm` |
| **Document** | `key -> JSON tree` | Varying shapes; nested data read whole | Cross-document joins | MongoDB, CouchDB |
| **Wide-column** | sparse rows, huge scale | Enormous write volume | Ad-hoc queries | Cassandra, HBase |
| **Graph** | nodes + edges | Deep relationships (see **10.6**) | Bulk aggregation | Neo4j |

There is a fifth thing often lumped in that is not really non-relational at all: **columnar analytical** engines like DuckDB and ClickHouse. They speak SQL and have a schema; they just store data by column instead of by row. We finish with one.

### The honest summary

A key-value store is a `dict` that outlives your process. A document store is a `dict` of `dict`s you can query inside. Most of what follows is that idea, made durable.

---

# Part 1: Key-value stores

## You already have one

The standard library ships a persistent key-value store: **`dbm`**. No server, no install, no dependency.

```
    with dbm.open(path, 'c') as kv:
                       ^^^
                       'r' read  'w' read/write  'c' create if needed  'n' always new
```

Keys and values are **`bytes`**. Strings are encoded to UTF-8 for you on the way in, but they always come back as `bytes` - a detail that catches everyone once.

In [ ]:
import dbm
import tempfile
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py105_"))     # never write into the notes
print("scratch:", WORK)

# A session store: the canonical key-value job.
with dbm.open(str(WORK / "sessions"), "c") as kv:
    kv["session:7f3a"] = "user=alice;role=admin"
    kv["session:91bc"] = "user=bob;role=viewer"

    print("lookup      :", kv["session:7f3a"])
    print("keys        :", kv.keys())
    print("how many    :", len(kv))
    print("in the store:", "session:91bc" in kv)

    del kv["session:91bc"]
    print("after delete:", len(kv))

print("\nNote the b'' prefixes - values come back as bytes, not str.")
print("backend chosen:", dbm.whichdb(str(WORK / "sessions")))

> ### Version note - `dbm.sqlite3` is the default from Python 3.13
>
> Historically `dbm` picked whichever C library was available (`gdbm`, `ndbm`) and fell back to `dbm.dumb`, a slow pure-Python format. Which one you got depended on the machine, so a file written on Linux might not open on Windows.
>
> **Python 3.13 added `dbm.sqlite3` and made it the default.** The file is now an SQLite database, which means it is portable across platforms and no longer needs a C library. The cell above confirms which backend you actually got.
>
> To be explicit rather than rely on the default, use `import dbm.sqlite3` and call `dbm.sqlite3.open()` directly.

## `shelve` - a `dbm` that stores Python objects

`dbm` only holds bytes. **`shelve`** wraps it with `pickle`, so values can be any picklable Python object - dicts, dataclasses, lists.

That convenience carries two warnings:

- 🔴 **`pickle` executes code on load.** Never open a shelf you did not write. This is a remote-code-execution hole, not a theoretical one.
- **The format is Python-specific.** No other language will read it. For interchange, use JSON.

In [ ]:
import shelve

shelf_path = str(WORK / "config")

with shelve.open(shelf_path) as shelf:
    shelf["retry_policy"] = {"attempts": 3, "backoff": [1, 2, 4]}
    shelf["feature_flags"] = {"new_search": True, "dark_mode": False}

# Reopened later - a different process would see exactly the same thing
with shelve.open(shelf_path) as shelf:
    policy = shelf["retry_policy"]
    print("a real dict, not bytes:", policy, type(policy).__name__)
    print("nested access        :", policy["backoff"][2])
    print("keys                 :", sorted(shelf.keys()))

### 🔴 The `shelve` writeback trap

This is the single most common `shelve` bug, and it fails **silently**.

A shelf only notices a change when you **assign to a key**. Mutating an object you read out of it changes your in-memory copy and nothing else — no error, no warning, and the next read returns the old value.

```
    shelf['policy']['attempts'] = 99      <- reads, mutates a copy, throws it away
    ^^^^^^^^^^^^^^^^                         the shelf never sees this
```

Three ways out, in order of preference: **read-modify-assign**, or `shelve.open(..., writeback=True)` (which caches every accessed entry in memory and writes them all back on close — convenient, and it can use a lot of memory), or don't store mutable structures.

In [ ]:
# ---- The trap ----
with shelve.open(shelf_path) as shelf:
    shelf["retry_policy"]["attempts"] = 99          # looks fine. is not.

with shelve.open(shelf_path) as shelf:
    print("after in-place mutation:", shelf["retry_policy"])
    print("  ^ still 3. The write was silently discarded.\n")

# ---- Fix 1: read, modify, assign back ----
with shelve.open(shelf_path) as shelf:
    policy = shelf["retry_policy"]      # read
    policy["attempts"] = 99             # modify
    shelf["retry_policy"] = policy      # assign back - THIS is what persists it

with shelve.open(shelf_path) as shelf:
    print("after read-modify-assign:", shelf["retry_policy"])

# ---- Fix 2: writeback=True ----
with shelve.open(shelf_path, writeback=True) as shelf:
    shelf["retry_policy"]["attempts"] = 5           # now this DOES stick

with shelve.open(shelf_path) as shelf:
    print("after writeback=True   :", shelf["retry_policy"])

## Redis - what a real key-value server adds

`dbm` and `shelve` are single-process, local files. A key-value **server** adds the things a cache actually needs:

| | `dbm`/`shelve` | Redis |
|---|---|---|
| shared between processes/machines | no | yes |
| expiry (TTL) | you build it | built in |
| atomic increment | you build it (and get it wrong) | `INCR` |
| data structures | bytes / pickle | lists, sets, sorted sets, hashes, streams |

The next cell probes what is reachable. Everything after it works either way.

In [ ]:
import os
import socket


def server_available(host: str, port: int, timeout: float = 0.5) -> bool:
    """Cheap check before paying a driver's connection timeout."""
    with socket.socket() as probe:
        probe.settimeout(timeout)
        try:
            probe.connect((host, port))
            return True
        except OSError:
            return False


HOST = os.environ.get("PYNOTES_DB_HOST", "127.0.0.1")
REDIS_PORT = int(os.environ.get("PYNOTES_REDIS_PORT", 56379))
MONGO_PORT = int(os.environ.get("PYNOTES_MONGO_PORT", 57017))
PG_PORT = int(os.environ.get("PYNOTES_PG_PORT", 55432))

HAVE_REDIS = server_available(HOST, REDIS_PORT)
HAVE_MONGO = server_available(HOST, MONGO_PORT)
HAVE_PG = server_available(HOST, PG_PORT)

for label, port, up in [
    ("Redis     ", REDIS_PORT, HAVE_REDIS),
    ("MongoDB   ", MONGO_PORT, HAVE_MONGO),
    ("PostgreSQL", PG_PORT, HAVE_PG),
]:
    print(f"{label} {HOST}:{port} ->", "reachable" if up else "not running")

if not (HAVE_REDIS or HAVE_MONGO or HAVE_PG):
    print(
        "\nNo servers. Every idea below still runs - on the stdlib and TinyDB.\n"
        "For the real thing:\n"
        '  docker compose -f "10 Database/docker/docker-compose.yml" up -d'
    )

In [ ]:
redis_client = None

if HAVE_REDIS:
    import redis

    redis_client = redis.Redis(host=HOST, port=REDIS_PORT, decode_responses=True)
    redis_client.delete("session:7f3a", "hits:/api/search", "cfg:api")

    # ---- TTL: the reason caches use Redis ----
    # NOTE: setex() is deprecated in redis-py 8; ex= on set() is the current form.
    redis_client.set("session:7f3a", "user=alice;role=admin", ex=60)
    print("get :", redis_client.get("session:7f3a"))
    print("ttl :", redis_client.ttl("session:7f3a"), "seconds left - it expires itself")

    # ---- Atomic counter: correct even with many writers ----
    for _ in range(3):
        hits = redis_client.incr("hits:/api/search")
    print("incr:", hits, "- no read-modify-write race, unlike a dict")

    # ---- A hash: one key holding fields ----
    redis_client.hset("cfg:api", mapping={"timeout": "30", "retries": "3"})
    print("hash:", redis_client.hgetall("cfg:api"))
else:
    print("Redis not running. The equivalent calls would be:")
    print("    r.set('session:7f3a', 'user=alice', ex=60)   # value that expires")
    print("    r.incr('hits:/api/search')                   # atomic counter")
    print("    r.hset('cfg:api', mapping={'timeout': '30'})  # hash")
    print()
    print("The stdlib has no TTL and no atomic increment. Building either")
    print("correctly across processes is exactly why Redis exists.")

---

# Part 2: Document stores

## Schema-on-write vs schema-on-read

| | Relational | Document |
|---|---|---|
| When is shape enforced? | **On write** - the database rejects bad rows | **On read** - your code copes with whatever it finds |
| Adding a field | `ALTER TABLE`, a migration | just write it |
| Guarantee | every row has every column | none |

That last line is the whole trade. The flexibility is real, and so is the fact that **nothing stops you writing `{"stat": "queued"}` when every other document says `"state"`.** The typo becomes a document that silently never matches a query.

> **The rule of thumb:** schema-on-read does not remove the schema. It moves it out of the database and into every piece of code that touches the data.

## You can build one on SQLite

SQLite has shipped the **JSON1** functions since 3.38 (and they are compiled in by default now), so a perfectly good document store is a two-column table:

```
    CREATE TABLE doc (id INTEGER PRIMARY KEY, body TEXT)
                                              ^^^^
                                              the whole JSON document

    json_extract(body, '$.owner.team')
                        ^^^^^^^^^^^^
                        a path: $ is the root, dots walk down
```

This is worth knowing well. It is often the right answer for "I need somewhere to put these varying payloads" without adding a whole server to your deployment.

In [ ]:
import json
import sqlite3

JOBS = [
    {"job": "reindex-search", "state": "queued", "retries": 2,
     "owner": {"team": "search", "oncall": "alice"}},
    {"job": "purge-cache", "state": "done", "retries": 0,
     "owner": {"team": "platform", "oncall": "bob"}},
    {"job": "send-digest", "state": "queued", "retries": 5,
     "owner": {"team": "growth", "oncall": "carol"},
     "schedule": "0 9 * * 1"},          # <- a field the others do not have
]

doc_db = sqlite3.connect(":memory:")
doc_db.execute("CREATE TABLE doc (id INTEGER PRIMARY KEY, body TEXT NOT NULL)")
doc_db.executemany(
    "INSERT INTO doc (body) VALUES (?)",
    [(json.dumps(d),) for d in JOBS],
)

print("-- reach into the document --")
for row in doc_db.execute(
    "SELECT id, json_extract(body,'$.job'), json_extract(body,'$.owner.team') FROM doc"
):
    print("   ", row)

print("\n-- a field only one document has --")
for job, schedule in doc_db.execute(
    "SELECT json_extract(body,'$.job'), json_extract(body,'$.schedule') FROM doc"
):
    note = "  <- key absent: NULL, not an error" if schedule is None else ""
    print(f"    {job:<16} {schedule!r}{note}")

print("\n-- query by a value inside the JSON --")
for row in doc_db.execute(
    "SELECT json_extract(body,'$.job') FROM doc WHERE json_extract(body,'$.state')=?",
    ("queued",),
):
    print("   ", row[0])

### Making it fast: index a JSON path

The query above works, but it must parse **every** document to evaluate the `WHERE`. That is a full scan.

A **generated column** materialises one JSON path as a real column, which can then be indexed like any other. `VIRTUAL` means it is computed on read and costs no storage; `STORED` writes it to disk and costs a little.

The cell below proves the index is used with `EXPLAIN QUERY PLAN` - `SEARCH ... USING INDEX` rather than `SCAN`.

In [ ]:
doc_db.execute(
    "ALTER TABLE doc ADD COLUMN state TEXT "
    "GENERATED ALWAYS AS (json_extract(body,'$.state')) VIRTUAL"
)
doc_db.execute("CREATE INDEX ix_doc_state ON doc(state)")

print("before the index existed, this was a full scan.")
print("plan now:")
for row in doc_db.execute(
    "EXPLAIN QUERY PLAN SELECT id FROM doc WHERE state='queued'"
):
    print("   ", row[-1])

print("\nand it queries like a normal column:")
for row in doc_db.execute("SELECT id, state FROM doc WHERE state=?", ("queued",)):
    print("   ", row)

print("\njson_each unrolls a nested object into rows:")
for row in doc_db.execute(
    "SELECT json_extract(d.body,'$.job'), j.key, j.value "
    "FROM doc d, json_each(d.body,'$.owner') j"
):
    print("   ", row)

## PostgreSQL `JSONB` - the middle ground most teams actually want

This is the pragmatic answer to "we need document flexibility": a relational database with a real document type, so **one** system gives you constraints and transactions *and* schema-less columns.

| Operator | Means | Example |
|---|---|---|
| `->` | get field, **as JSON** | `body->'owner'` |
| `->>` | get field, **as text** | `body->>'state'` |
| `#>>` | get by **path**, as text | `body#>>'{owner,team}'` |
| `@>` | **contains** | `body @> '{"state":"queued"}'` |

`JSONB` is parsed and stored in a binary form (so key order is not preserved and duplicate keys are dropped) — which is what makes it indexable. A **GIN** index over the whole column supports containment queries on *any* key, without naming it in advance.

In [ ]:
pg_conn = None

if HAVE_PG:
    import psycopg

    pg_conn = psycopg.connect(
        host=HOST, port=PG_PORT,
        user=os.environ.get("PYNOTES_PG_USER", "learner"),
        password=os.environ.get("PYNOTES_PG_PASSWORD", "learnpython"),
        dbname=os.environ.get("PYNOTES_PG_DB", "notes"),
        connect_timeout=5,
    )
    pg_conn.execute("DROP TABLE IF EXISTS doc")
    pg_conn.execute(
        "CREATE TABLE doc ("
        "  id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,"
        "  body JSONB NOT NULL)"
    )
    with pg_conn.cursor() as cur:
        for d in JOBS:
            cur.execute("INSERT INTO doc (body) VALUES (%s)", (json.dumps(d),))
    pg_conn.commit()

    print("->>  field as text     :",
          pg_conn.execute("SELECT id, body->>'job' FROM doc ORDER BY id").fetchall())
    print("#>>  nested path       :",
          pg_conn.execute("SELECT body#>>'{owner,team}' FROM doc ORDER BY id").fetchall())
    print("@>   containment       :",
          pg_conn.execute("SELECT body->>'job' FROM doc WHERE body @> %s",
                          ('{"state":"queued"}',)).fetchall())

    pg_conn.execute("CREATE INDEX ix_doc_gin ON doc USING GIN (body)")
    pg_conn.commit()
    print("\nGIN index created - now ANY key can be searched by containment,")
    print("without having declared it up front.")

    print("\njsonb_set updates inside the document:")
    print("   ", pg_conn.execute(
        "SELECT body->>'job', jsonb_set(body,'{retries}','9')->>'retries' "
        "FROM doc ORDER BY id LIMIT 1").fetchone())
else:
    print("PostgreSQL not running. The equivalent SQL:")
    print("    CREATE TABLE doc (id int PRIMARY KEY, body JSONB);")
    print("    SELECT body->>'job' FROM doc WHERE body @> '{\"state\":\"queued\"}';")
    print("    CREATE INDEX ix_doc_gin ON doc USING GIN (body);")
    print()
    print("The SQLite version above teaches the same idea and runs anywhere.")

## MongoDB - the dedicated document database

Mongo's model is **collections of documents**. There is no schema, `_id` is generated for you, and queries are written as dictionaries rather than as a query language.

| SQL | MongoDB |
|---|---|
| `SELECT * FROM job WHERE state='queued'` | `col.find({"state": "queued"})` |
| `SELECT job FROM ...` | `col.find({...}, {"job": 1})` |
| `GROUP BY state` | `col.aggregate([{"$group": {...}}])` |
| `CREATE INDEX` | `col.create_index("state")` |

Nested fields use **dotted paths**: `{"owner.team": "search"}`.

🔴 `find()` returns a **cursor**, not a list. It is lazy, it can only be consumed once, and printing it shows a cursor object rather than your data.

In [ ]:
mongo_client = None

if HAVE_MONGO:
    from pymongo import MongoClient

    user = os.environ.get("PYNOTES_MONGO_USER", "learner")
    pwd = os.environ.get("PYNOTES_MONGO_PASSWORD", "learnpython")
    mongo_client = MongoClient(
        f"mongodb://{user}:{pwd}@{HOST}:{MONGO_PORT}/?authSource=admin",
        serverSelectionTimeoutMS=3000,
    )
    jobs = mongo_client.notes.jobs
    jobs.drop()                                  # re-runnable
    jobs.insert_many([dict(d) for d in JOBS])    # copy: insert_many adds _id in place

    print("find (exclude _id)  :")
    for d in jobs.find({"state": "queued"}, {"_id": 0, "job": 1, "retries": 1}):
        print("   ", d)

    print("\ndotted path into nested object:")
    for d in jobs.find({"owner.team": "search"}, {"_id": 0, "job": 1}):
        print("   ", d)

    print("\ncomparison operator ($gt):")
    for d in jobs.find({"retries": {"$gt": 1}}, {"_id": 0, "job": 1, "retries": 1}):
        print("   ", d)

    print("\naggregate - the GROUP BY equivalent:")
    for d in jobs.aggregate([
        {"$group": {"_id": "$state", "n": {"$sum": 1}}},
        {"$sort": {"_id": 1}},
    ]):
        print("   ", d)

    jobs.create_index("state")
    print("\nindexes:", sorted(jobs.index_information()))
else:
    print("MongoDB not running. The equivalent calls:")
    print('    jobs.find({"state": "queued"}, {"_id": 0})')
    print('    jobs.find({"owner.team": "search"})        # dotted path')
    print('    jobs.find({"retries": {"$gt": 1}})')
    print('    jobs.aggregate([{"$group": {"_id": "$state", "n": {"$sum": 1}}}])')
    print()
    print("TinyDB below runs the same shape of query with no server at all.")

## TinyDB - a document store with no server, ever

Pure Python, one JSON file, a query API deliberately close to Mongo's. It is not for production, and it is genuinely useful for small tools, prototypes and tests — and it means the document-query idea below runs on any machine.

```
    Job = Query()
    tdb.search(Job.state == 'queued')
               ^^^^^^^^^^^^^^^^^^^^^
               builds a query object; == is overloaded, not evaluated (see 5.1)
```

In [ ]:
from tinydb import Query, TinyDB

with TinyDB(WORK / "jobs.json") as tdb:
    tdb.truncate()                       # re-runnable
    tdb.insert_multiple(JOBS)

    Job = Query()

    print("state == 'queued':")
    for d in tdb.search(Job.state == "queued"):
        print("   ", d["job"])

    print("\nnested - owner.team == 'search':")
    for d in tdb.search(Job.owner.team == "search"):
        print("   ", d["job"])

    print("\nretries > 1:")
    for d in tdb.search(Job.retries > 1):
        print("   ", d["job"], d["retries"])

    print("\ndocuments that HAVE a schedule field:")
    for d in tdb.search(Job.schedule.exists()):
        print("   ", d["job"], "->", d["schedule"])

    print("\ncount:", tdb.count(Job.state == "queued"))

---

## 🔴 What you gave up

The relational database was doing work for you that is easy to miss until it stops.

The next cell writes three documents that **any** document store accepts happily:

- one with `stat` instead of `state` — a typo
- one where `retries` is the string `"three"` instead of a number
- one referring to a team that does not exist

In a relational table, a `CHECK` constraint, a column type and a foreign key would have rejected all three **at write time**. Here they are stored, and you find out later — usually as a query that silently returns fewer rows than it should.

In [ ]:
BAD = [
    {"job": "rotate-logs", "stat": "queued", "retries": 0},        # typo: stat
    {"job": "warm-cache", "state": "queued", "retries": "three"},   # str, not int
    {"job": "ship-report", "state": "queued", "retries": 1,
     "owner": {"team": "does-not-exist"}},                          # dangling ref
]

with TinyDB(WORK / "bad.json") as tdb:
    tdb.truncate()
    tdb.insert_multiple(JOBS + BAD)
    Job = Query()

    queued = tdb.search(Job.state == "queued")

    # What a person reading the raw documents would call "queued" - which
    # includes the one whose key is misspelled.
    intended = sum(
        1 for d in JOBS + BAD
        if d.get("state") == "queued" or d.get("stat") == "queued"
    )
    print("documents that LOOK queued to a human:", intended)
    print("documents the query actually returns :", len(queued))
    print("  ^ 'rotate-logs' is invisible - its key is 'stat', not 'state'")
    print()

    # And the type confusion only shows up when you compute with it
    total = 0
    for d in tdb.search(Job.state == "queued"):
        try:
            total += d["retries"]
        except TypeError as exc:
            print(f"summing retries failed on {d['job']!r}: {exc}")
    print("partial total:", total)

print()
print("None of this is a bug in the database. It did exactly what it promised:")
print("store whatever you give it. The validation has to live in your code -")
print("see 5.3 (dataclasses) and 16 (type checking) for where to put it.")

## Eventual consistency and CAP, without the hand-waving

Once data lives on **more than one machine**, a network failure forces a choice. That is all CAP says.

> A distributed store, when the network between its nodes breaks, must choose:
> **refuse to answer** (stay consistent) or **answer with possibly-stale data** (stay available). It cannot do both.

**A concrete example.** You update your profile picture. The write lands on the European node. A friend in Australia reads from the Sydney node a moment later and still sees the old one. Ten seconds on, they see the new one. That is **eventual consistency**: not wrong, just not immediate.

Where it matters:

| Data | Stale for 10 seconds is... |
|---|---|
| Profile picture, like count | fine |
| Inventory on a checkout page | uncomfortable |
| Account balance in a transfer | unacceptable |

**On one machine, none of this applies.** A single-node MongoDB is as consistent as a single-node PostgreSQL. "NoSQL is eventually consistent" is a statement about *distribution*, not about not being relational — and modern MongoDB supports multi-document ACID transactions on a replica set.

## DuckDB - the one that is not non-relational at all

It speaks SQL and it has a schema. What differs is **physical layout**: DuckDB stores data by *column* rather than by *row*.

```
    row store    [id|svc|ms] [id|svc|ms] [id|svc|ms]   <- SQLite, Postgres, MySQL
    column store [id id id] [svc svc svc] [ms ms ms]   <- DuckDB, ClickHouse
```

`SELECT avg(ms) FROM ev` touches one column. A row store reads every byte of every row; a column store reads only `ms`. On a hundred million rows that is the whole game.

The right mental split: **row store for transactions** (fetch and update whole records), **column store for analytics** (aggregate one or two columns across everything). DuckDB is in-process like SQLite — no server — which makes it the natural analytical companion.

In [ ]:
import duckdb

an = duckdb.connect()          # in-process, in-memory - no server
an.execute("CREATE TABLE ev (day DATE, svc VARCHAR, ms INTEGER)")
an.execute(
    "INSERT INTO ev VALUES "
    "('2026-08-01','api',12),('2026-08-01','api',30),"
    "('2026-08-02','db',7),('2026-08-02','api',19),('2026-08-02','db',41)"
)

print("ordinary SQL aggregate:")
for row in an.execute(
    "SELECT svc, count(*) AS n, round(avg(ms),1) AS avg_ms "
    "FROM ev GROUP BY svc ORDER BY svc"
).fetchall():
    print("   ", row)

print("\nwindow function - a running total per service:")
for row in an.execute(
    "SELECT svc, ms, sum(ms) OVER (PARTITION BY svc ORDER BY ms) AS running "
    "FROM ev ORDER BY svc, ms"
).fetchall():
    print("   ", row)

an.close()
print("\nSame SQL you already know. Different storage, different strengths.")

---

## Choosing

| You need... | Reach for |
|---|---|
| Related data, constraints, transactions | **Relational** (10.1-10.4). Start here. |
| A cache, a session store, a rate limiter | **Key-value** - Redis; `dbm` if single-process |
| Varying payloads, read whole, few joins | **Document** - Postgres `JSONB` first, Mongo if you outgrow it |
| Flexible fields *and* constraints | **PostgreSQL `JSONB`** - genuinely both |
| Deep relationship traversal | **Graph** - see **10.6** |
| Aggregates over enormous tables | **Columnar** - DuckDB, ClickHouse |
| A small local tool, no server | `sqlite3` + JSON1, or TinyDB |

**The most common real-world mistake is reaching for a document store to avoid writing a schema.** The schema does not go away. It moves into your code, where nothing enforces it.

In [ ]:
# ---- tidy up: close every connection and remove the scratch directory ----
import shutil

doc_db.close()

if redis_client is not None:
    redis_client.delete("session:7f3a", "hits:/api/search", "cfg:api")
    redis_client.close()
    print("redis   : keys removed, client closed")

if mongo_client is not None:
    mongo_client.notes.jobs.drop()
    # 🔴 An unclosed MongoClient raises ResourceWarning at interpreter exit.
    mongo_client.close()
    print("mongo   : collection dropped, client closed")

if pg_conn is not None:
    pg_conn.execute("DROP TABLE IF EXISTS doc")
    pg_conn.commit()
    pg_conn.close()
    print("postgres: table dropped, connection closed")

shutil.rmtree(WORK, ignore_errors=True)
print("scratch : removed ->", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Mutating a value read out of a `shelve` shelf.** The write is discarded silently. Read, modify, assign back - or open with `writeback=True`.
2. 🔴 **Unpickling a shelf you did not create.** `pickle` executes code on load.
3. 🔴 **Assuming a document store validates anything.** A typo in a key produces a document that never matches a query, and no error anywhere.
4. **Expecting `str` back from `dbm`.** Keys and values are always `bytes`.
5. **Treating `find()` as a list.** It is a lazy cursor, consumable once.
6. **Querying a JSON field without indexing it.** Every document gets parsed - a full scan. Use a generated column (SQLite) or a GIN index (PostgreSQL).
7. **Reaching for MongoDB to avoid writing a schema.** PostgreSQL `JSONB` usually gives the flexibility without giving up constraints and transactions.
8. **Quoting CAP about a single-node database.** It is about distributed systems; on one machine it does not apply.
9. **Using `setex()` with redis-py 8.** Deprecated - use `set(key, value, ex=seconds)`.

## Best Practices

- Start relational. Move a *specific* problem to a specialised store when you have measured that it is a problem.
- Prefer PostgreSQL `JSONB` over a second database when you need flexible fields.
- Validate documents in code on the way in - dataclasses (**5.3**) or pydantic.
- Give keys a namespace convention: `session:<id>`, `cfg:<service>`.
- Always set a TTL on cache entries; a cache without expiry is a memory leak.
- Index the JSON paths you actually query, and check the plan to confirm.
- Close clients explicitly - `MongoClient` and `Redis` both hold sockets.
- Keep credentials in the environment, never in the notebook (**10.2**).

## Practice Exercises

Try these before moving on.

1. Write a `get`/`set`/`delete` cache class backed by `dbm.sqlite3`, storing an expiry timestamp alongside each value so it can implement its own TTL.
2. Reproduce the `shelve` writeback trap, then fix it both ways. Which do you prefer, and why might `writeback=True` be a bad idea for a large shelf?
3. Add a second generated column and index for `$.owner.team` in the SQLite document store, and confirm with `EXPLAIN QUERY PLAN` that the index is used.
4. Start the stack. Insert 1000 documents into PostgreSQL `JSONB`, query with `@>` before and after creating the GIN index, and compare `EXPLAIN ANALYZE`.
5. Take the `BAD` documents and write a `validate(doc)` function using a dataclass (**5.3**) that rejects all three. Where should it be called?
6. Express `SELECT state, COUNT(*) FROM job GROUP BY state` three ways: SQL, a Mongo aggregation pipeline, and plain Python over TinyDB results.
7. Load a CSV of at least 100k rows into both SQLite and DuckDB and time `SELECT avg(col)`. Explain the difference.